In [4]:
import numpy as np
import torch
import torch.nn as nn
import gymnasium as gym
import metaworld
import imageio, collections, json, os, time
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor
from stable_baselines3.common.callbacks import BaseCallback

TASK_NAME = "peg-insert-side-v3"
SUCCESS_KEY = "success"

def make_env(seed=0, render_mode=None):
    return gym.make("Meta-World/MT1", env_name=TASK_NAME, seed=seed, render_mode=render_mode,  camera_name="corner2")

_probe = make_env(seed=0)
DT = getattr(_probe.unwrapped, "dt", _probe.unwrapped.model.opt.timestep)
RAW_OBS_DIM = _probe.observation_space.shape[0]
_probe.close()
# print(f"DT={DT}, raw obs dim={RAW_OBS_DIM}, feature dim={FEATURE_DIM}")

class Time2SuccessModel(nn.Module):
    """dropout=0.2 must match whatever the saved checkpoints were trained with,
    or load_state_dict fails on layer-index mismatch."""
    def __init__(self, obs_dim, hidden=256, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

In [2]:
# ============================================================
# SHARED FEATURE BUILDER — identical copy in all three notebooks.
# If you change it here, change it EVERYWHERE. A mismatch in column
# order fails silently: the model receives numbers in the wrong slots
# and returns plausible-looking garbage without raising.
# ============================================================
HAND_POS_IDX = [0, 1, 2]
GRIPPER_IDX  = 3
PEG_POS_IDX  = [4, 5, 6]
GOAL_POS_IDX = [36, 37, 38]
GRIPPER_CLOSED_THRESHOLD = 0.6
FEATURE_DIM = 49

def build_features(obs):
    """39-dim raw obs -> 49-dim feature vector.

    Columns:
       0-38  raw obs (unchanged)
      39-41  hand->peg vector
      42-44  peg->goal vector
         45  |hand->peg|
         46  |peg->goal|
         47  gripper openness
         48  holding flag (0/1)
    """
    obs = np.asarray(obs, dtype=np.float32)
    hand = obs[HAND_POS_IDX]
    peg  = obs[PEG_POS_IDX]
    goal = obs[GOAL_POS_IDX]
    gripper = obs[GRIPPER_IDX]

    hand_to_peg_vec = peg - hand
    peg_to_goal_vec = goal - peg
    hand_to_peg = np.linalg.norm(hand_to_peg_vec)
    peg_to_goal = np.linalg.norm(peg_to_goal_vec)
    holding = float(gripper < GRIPPER_CLOSED_THRESHOLD and hand_to_peg < 0.06)

    return np.concatenate([
        obs,
        hand_to_peg_vec,
        peg_to_goal_vec,
        [hand_to_peg, peg_to_goal, gripper, holding],
    ]).astype(np.float32)

In [6]:
import glob, os, imageio, numpy as np
from stable_baselines3 import SAC
from IPython.display import Video

# needs: make_env, SUCCESS_KEY, PEG_POS_IDX defined in this kernel (cells 3-4 of nb3)

CKPT_DIR = "./checkpoints/peg_pure_t2s_v5_difference"
ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "ckpt_*.zip")),
               key=lambda p: int(os.path.basename(p).split("_")[1].split(".")[0]))
print("available:", [os.path.basename(c) for c in ckpts])

p = SAC.load(ckpts[-1])          # most recent
e = make_env(seed=0, render_mode="rgb_array")
obs, _ = e.reset()
frames, ss = [], None
peg_start = obs[PEG_POS_IDX].copy()
peg_total = 0.0
last_peg = peg_start.copy()

for t in range(500):
    frames.append(e.render())
    a, _ = p.predict(obs, deterministic=True)
    obs, _, term, trunc, info = e.step(a)
    peg_total += np.linalg.norm(obs[PEG_POS_IDX] - last_peg)
    last_peg = obs[PEG_POS_IDX].copy()
    if info.get(SUCCESS_KEY, 0) and ss is None:
        ss = t
    if term or trunc:
        break
e.close()

imageio.mimsave("progress_check_final.mp4", frames, fps=20)
print(f"checkpoint: {os.path.basename(ckpts[-1])}")
print(f"success_step: {ss}")
print(f"peg displacement from start: {np.linalg.norm(last_peg - peg_start):.4f} m")
print(f"peg path length: {peg_total:.4f} m")

available: ['ckpt_2004.zip', 'ckpt_4002.zip', 'ckpt_500004.zip', 'ckpt_1000002.zip', 'ckpt_1500000.zip', 'ckpt_2000004.zip', 'ckpt_2500002.zip', 'ckpt_3000000.zip']
checkpoint: ckpt_3000000.zip
success_step: None
peg displacement from start: 0.0052 m
peg path length: 0.0096 m
